In [1]:
import tiktoken
import numpy as np
# from senten

In [2]:
def softmax(x, axis= 0):
    # subtract max for numerical stability

    # across rows
    x_shifted = x - np.max(x, axis = axis, keepdims=True)
    
    exp_x = np.exp(x_shifted)
    
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [3]:
# axis= 0 # column
# axis= 1 # row

In [4]:
def self_attention(query , key, value):
    # q, k, v are np arrays and are of same order
    # writing codea asuming they are horizontal matrices
    dk = key.shape[1]
    a = (query @ key.T) / np.sqrt(dk) # dot product but for a number of words will be orchestrsted by cross product

    a = softmax(a , axis = 1)
    
    b = a @ value
    
    return b
    

In [5]:
class MultiHeadAttention:

    def __init__(self, embed_dim, num_heads, masked = False):
        self.embed_dim = embed_dim 
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.masked = masked

        self.WQ = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WK = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WV = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]

        self.WO = np.random.randn(embed_dim, embed_dim)

    def self_attention(self,query , key, value):
        score = (query @ key.T) / np.sqrt(self.head_dim)
        # print("Before Mask : " ,score)
        if (self.masked):
            # print("Masked is turned true")
            r,c = score.shape
            # print ("r= ", r , " c = ", c)
            for i in range(r):
                for j in range(i+1, c):
                    score[i][j] = -np.inf
            # print("After Mask : ", score)



        attention = softmax(score , axis = 1)
        # print("After Softmax: ", attention)
        
        b = attention @ value
        return b
            
    
    def forward(self, query, key, value):
        heads =  []
        for i in range(self.num_heads):

            q = query @ self.WQ[i]
            k = key @ self.WK[i]
            v = value @ self.WV[i]
            head = self.self_attention(q, k, v)
            heads.append(head)

        context_awared = np.concatenate(heads, axis = 1)

        output = context_awared @ self.WO # without this step all the heads will be separate. But this combines the information of all the heads
        return output

positional encoder

In [6]:
# based on formula

def positional_encoder_f(pos, embedding_dimension):

    if (embedding_dimension % 2 != 0 ):
        raise ValueError("Embedding dimension must be even.")
    encoded_pos = []    
    for i in range (int((embedding_dimension / 2))) :
        a = np.sin(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        b = np.cos(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        encoded_pos += [a,b]

    return encoded_pos

In [7]:
def get_positional_encoding_for_fixed_number_of_words(number_of_words, embedding_dimension):
    encoded_positions = []
    for i in range (number_of_words):
        encoded_positions += [positional_encoder_f(i,embedding_dimension)]
        # print(encoded_positions, "\n")

    # print("\n\n", encoded_positions)
    return np.vstack([encoded_positions])
    
    

layer normalization


In [8]:
class LayerNorm:
    def __init__ (self, embed_dim):

        self.gamma = np.ones(embed_dim) # initializing to 1 because initially no scaling 
        self.beta = np.zeros(embed_dim) # initializing to 0 because initially no shifting

    def forward (self, x):
        m = np.mean(x, axis = 1, keepdims=True)
        s = np.std(x, axis = 1, keepdims=True)

        normalized = (x - m ) / (s + 1e-5) # incase preventing divison by zero

        return self.gamma * normalized + self.beta

feed forward network


In [9]:
import torch
from torch import nn
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [10]:
class FeedForward(nn.Module):
    def __init__(self ,embedding_dimension, hidden_dimension):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(embedding_dimension, hidden_dimension)
        self.linear2 = nn.Linear(hidden_dimension, embedding_dimension)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

In [11]:
class EncoderBlock:

    def __init__(self, embed_dim, num_heads, d_ff):
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.attention = MultiHeadAttention(self.embed_dim, self.num_heads)

        self.norm = LayerNorm(embed_dim)
        self.norm2 = LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim, d_ff )

    

    def forward(self, x):

        # x is with positional encoding information

        attention_output = self.attention.forward(x,x,x) # self attention

        x = x + attention_output

        x = self.norm.forward(x) # learnable beta and gamma

        x_tensor = torch.tensor(x, dtype=torch.float32)

        ff_o = self.ffn.forward(x_tensor)

        x = x + (ff_o).detach().numpy()

        x = self.norm2.forward(x) # learnable beta gamma 2

        return x


In [12]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4") # it uses BPE
print(enc.n_vocab)
tokens = (enc.encode("Hello Mangesh here i am  <|endoftext|>", allowed_special={"<|endoftext|>"}))


100277


In [13]:
np.random.randn(2,4)[0]

array([ 0.62306624,  0.07537821, -1.19619303, -0.67215437])

In [14]:
class Embeddings:

    def __init__(self, vocab_size = 100277, embed_dim = 512):

        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embedding_matrix = np.random.randn(vocab_size, embed_dim)
        # print("Shaoe of embeddig martrix: " ,self.embedding_matrix.shape)
        


    # vocab_size = enc.n_vocab # it has inbuilt special token <|endoftext|>

    def get_embedidngs_with_pe(self, tokens):
        embeddings = []
        token_count = len(tokens)
        for t in tokens:
            embeddings += [self.embedding_matrix[t]]
        embeddings = np.stack(embeddings, axis = 0)
        pe = get_positional_encoding_for_fixed_number_of_words(token_count, self.embed_dim)

        return embeddings + pe
    


### Decoder Block

In [15]:
class DecoderBlock:

    def __init__(self, embed_dim, num_heads, d_ff):
        
        self.masked_attention = MultiHeadAttention(embed_dim, num_heads, masked = True)

        self.norm1 = LayerNorm(embed_dim)

        self.cross_attention = MultiHeadAttention(embed_dim, num_heads, masked = False)
        self.norm2  = LayerNorm(embed_dim)

        self.ffn = FeedForward(embed_dim, d_ff)
        self.norm3 = LayerNorm(embed_dim)
    

    def forward (self, decoder_input, encoder_output):


        masked_output = self.masked_attention.forward(decoder_input, decoder_input, decoder_input)
        # print("masked op:", masked_output[-1][10])
        x = decoder_input + masked_output

        # x = self.norm1.forward(x)

        cross_att_op = self.cross_attention.forward(x, encoder_output, encoder_output)
        # print("Cross att: ", cross_att_op[-1][:10])
        x = cross_att_op + x

        # x = self.norm2.forward(x)

        x_tensor = torch.tensor(x, dtype=torch.float32)

        ff_o = self.ffn.forward(x_tensor)

        x = x + (ff_o).detach().numpy()

        # x = self.norm3.forward(x)
        # print("FInal decoder block op:" ,x.shape)
        return x
        


In [16]:
class Encoder:

    def __init__(self, num_layers, embed_dim, num_heads, d_ff):

        self.layers  = [EncoderBlock(embed_dim, num_heads, d_ff) for _ in range(num_layers)]

    def forward(self, x):

        for layer in self.layers:
            x = layer.forward(x)

        return x
    
class Decoder:
    def __init__(self,num_layers ,embed_dim, num_heads, d_ff):

        self.layers = [DecoderBlock(embed_dim, num_heads, d_ff) for _ in range(num_layers)]

    def forward (self, x, encoder_output):
        for layer in self.layers:
            x = layer.forward(x, encoder_output)

        return x
    


In [17]:
class OutputProj(nn.Module):
    def __init__(self, embed_dim, vocab_size):
        super().__init__()

        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
            
        x = self.linear(x)
        return x

In [ ]:
class Transformer:
    def __init__(self, enc_layers, embed_dim, enc_heads, d_enc_ff, dec_layers, dec_heads, d_dec_ff, vocab_size):
        
        self.encoder = Encoder(num_layers = enc_layers, embed_dim = embed_dim, num_heads = enc_heads, d_ff= d_enc_ff )

        self.decoder = Decoder(num_layers = dec_layers, embed_dim = embed_dim, num_heads = dec_heads, d_ff= d_dec_ff )

        self.output_proj = OutputProj(embed_dim=embed_dim, vocab_size= vocab_size)

        self.embedding = Embeddings(vocab_size, embed_dim)

        self.tokenizer = tiktoken.encoding_for_model('gpt-4')\
        
        self.special_tokens = self.tokenizer.special_tokens_set


    #inference

    def forward_inference(self, x): # x is tokens

        x = self.embedding.get_embedidngs_with_pe(x)

        encoder_output = self.encoder.forward(x)

        EOS_TOKEN = 100257

        generated = [EOS_TOKEN]
        MAX_LEN = 100
        for _ in range(MAX_LEN):
            
            dec_input = self.embedding.get_embedidngs_with_pe(generated)
            # print("decoder input: ", dec_input.shape)
            decoder_output = self.decoder.forward(dec_input, encoder_output)

            # print("decoderOutput: " , decoder_output.shape)

            logits = self.output_proj.forward(decoder_output)
            
            last_logits = logits[-1]

            values, indices = torch.topk(last_logits, 5)
            if decoder_output.shape[0] > 1:
                print("Checking: ", np.linalg.norm(decoder_output[-1] - decoder_output[-2]))
            

            # print("topk: ", topk )

            predicted_id = torch.argmax(last_logits).item()
            
            # print("logits.shape)", logits.shape)
            # print("predicted_id.shape: ",predicted_id.shape)
            # print("predicted_id: ",predicted_id)
            # print("predicted_id item : ", predicted_id.item())

            if (predicted_id == EOS_TOKEN): # preicted_id is a tensor
                break
            
            
            generated += [predicted_id]
            # print("Generated:", generated)

            # print("Last embedding:")
            # print(dec_input[-1][:10])

            # print("Last decoder output:")
            # print(decoder_output[-1][:10])

            next_word = self.tokenizer.decode([generated[-1]])
            print("next_word: ", next_word)          

In [19]:
transformer_obj = Transformer(6, 512, 8, 512, 6, 8, 512, 100277)

In [20]:
inputt = transformer_obj.tokenizer.encode("The cat is great ")
transformer_obj.forward_inference(inputt)


next_word:  iß
Checking:  1.1586307824277853e+17
next_word:  iß
Checking:  8.402564689995115e+16
next_word:  iß
Checking:  2296730535323.6772
next_word:  iß
Checking:  6.7583539732362216e+16
next_word:  iß
Checking:  20286016858985.508
next_word:  iß
Checking:  1860551376700.099
next_word:  iß
Checking:  60849395.206777655
next_word:  iß
Checking:  33379771054.10396
next_word:  iß
Checking:  3.912580331326736e+16
next_word:  iß
Checking:  60431133945.717865
next_word:  iß
Checking:  68790230573.14542
next_word:  iß
Checking:  3.2577275368724188e+16
next_word:  iß
Checking:  1134468466428.4287
next_word:  iß
Checking:  71752795.18244942
next_word:  iß
Checking:  6076305770.167897
next_word:  iß
Checking:  19105304.386145268
next_word:  iß
Checking:  334891629292110.94
next_word:  iß
Checking:  76668921499.46877
next_word:  iß
Checking:  4.632348672182929e+16
next_word:  iß
Checking:  147752603894944.12
next_word:  iß
Checking:  1829008811017.6943
next_word:  iß
Checking:  4.940942868446